In [ ]:
# Figure factory for the heart single-cell atlas.
import os, glob
import scanpy as sc
import anndata as ad
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch

In [ ]:
input_dir = "/Users/yanjun/Desktop/PQE_thesis/scripts/chapter5/"

In [ ]:
sc.set_figure_params(figsize=(3, 3),  facecolor="white",
    frameon=True,   dpi = 300, format = 'png')
sc.settings.figdir = "/Users/yanjun/Desktop/PQE_thesis/Chapter5_Figures"
sns.set_theme()
torch.set_float32_matmul_precision("high")

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

In [ ]:
adata = sc.read('/Users/yanjun/Desktop/PQE_thesis/scripts/chapter5/qc3_scvi.h5ad')

# sc0.4 ; sn0.5

In [ ]:
adata

In [ ]:
sample_counts = (adata.obs.groupby('sample')['project_id']
                 .first()
                 .value_counts())

total = sample_counts.sum()
labels = [f'{site} ({n/total*100:.1f}%, n={n})' 
         for site, n in sample_counts.items()]

fig, ax = plt.subplots(figsize=(10 , 6))
wedges, _ = ax.pie(sample_counts.values, startangle=90)
ax.legend(wedges, labels, title='project_id',
         loc='center left', bbox_to_anchor=(1, 0.5))
ax.set(aspect='equal')
plt.tight_layout()
plt.savefig("project_id.png", dpi=300) 
plt.show()

In [ ]:
sample_counts = (adata.obs.groupby('sample')['anatomical_site']
                 .first()
                 .value_counts())

total = sample_counts.sum()
labels = [f'{site} ({n/total*100:.1f}%, n={n})' 
         for site, n in sample_counts.items()]

fig, ax = plt.subplots(figsize=(8, 6))
wedges, _ = ax.pie(sample_counts.values, startangle=90)
ax.legend(wedges, labels, title='anatomical site',
         loc='center left', bbox_to_anchor=(1, 0.5))
ax.set(aspect='equal')
plt.tight_layout()
plt.savefig("anatomical_site.png.png", dpi=300) 
plt.show()

In [ ]:
sample_counts = (adata.obs.groupby('sample')['age']
                 .first()
                 .value_counts())

total = sample_counts.sum()
labels = [f'{site} ({n/total*100:.1f}%, n={n})' 
         for site, n in sample_counts.items()]

fig, ax = plt.subplots(figsize=(8, 6))
wedges, _ = ax.pie(sample_counts.values, startangle=90)
ax.legend(wedges, labels, title='age groups',
         loc='center left', bbox_to_anchor=(1, 0.5))
ax.set(aspect='equal')
plt.tight_layout()
plt.savefig("age.png", dpi=300) 
plt.show()

In [ ]:
sample_counts = (adata.obs.groupby('sample')['Disease']
                 .first()
                 .value_counts())

total = sample_counts.sum()
labels = [f'{site} ({n/total*100:.1f}%, n={n})' 
         for site, n in sample_counts.items()]

fig, ax = plt.subplots(figsize=(8, 6))
wedges, _ = ax.pie(sample_counts.values, startangle=90)
ax.legend(wedges, labels, title='Disease',
         loc='center left', bbox_to_anchor=(1, 0.5))
ax.set(aspect='equal')
plt.tight_layout()
plt.savefig("Disease.png.png", dpi=300) 
plt.show()

In [ ]:
sc.pl.umap(adata, color = 'leiden_res0_4', size = 30, )


In [ ]:
cluster_map = {
    '0': "heart_sc_Mac0",
    '1': "heart_sc_Mac1",
    '2': "heart_sc_Mac2",
    '3': "heart_sc_Mac3",
    '4': "heart_sc_Mac4",
    '5': "heart_sc_Mac5",
    '7': "heart_sc_Mac5",
    '8': "heart_sc_Mac2"
}

adata.obs["anno_layer1"] = adata.obs["leiden_res0_4"].map(cluster_map)

In [ ]:
sc.pl.umap(adata, color = 'anno_layer1', size = 30, 
save = 'sc_umap.png')

In [ ]:
adata

In [ ]:
del adata.obs['age']

In [ ]:
info = pd.read_csv("/Users/yanjun/Desktop/PQE_thesis/scripts/chapter5/heart_sc_clean_.csv").drop_duplicates(subset=["sample_id"])

meta = info.set_index("sample_id")
new_cols = meta.columns.difference(adata.obs.columns)

adata.obs = adata.obs.join(meta[new_cols], on="sample")

In [ ]:
sc.pl.umap(adata, 
          color = ['G2M_score'  , 'pct_counts_mt'], size = 30,
           color_map = 'RdBu_r', ncols = 1, save = 'sc_umap_qc.png'
          )

In [ ]:
sc.pl.umap(adata, 
          color = ['HMOX1'  , 'HBA1', 'age','anno_layer1'], size = 30,
           color_map = 'RdBu_r', ncols = 2, 
        
           save = 'hmx.png'
          )

In [ ]:
sc.pl.umap(adata, 
          color = ['Disease'  , 'anatomical_site', 'age', 'anno_layer1'], size = 30,
           color_map = 'RdBu_r', ncols = 1, save = 'sc_umap_metadata.png'
          )

In [ ]:
adata.write('qc3_scvi.h5ad')

In [ ]:
sc.pl.umap(adata, 
          color = ['LYVE1', 'MRC1', 'SELENOP', 'F13A1', 'FOLR2','C1QA', 'CD68',
                  'C1QC',], size = 30,
           color_map = 'RdBu_r', ncols = 2, save = 'umap_lyve1.png'
          )

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=["LYVE1", "MRC1", "F13A1", "FOLR2", 'NRP1', 'C1QA', 'CD68',  'C1QC',
    'STIL', 'HMOX1', 'HBB'],
    groupby="anno_layer1",
    standard_scale="var",
    cmap = 'RdBu_r',
    swap_axes=True,
    save = 'dotplot.png'
)

In [ ]:
lyve1_genes = ["MRC1", "F13A1", "LYVE1", "FOLR2",]

sc.tl.score_genes(
    adata,
    gene_list=lyve1_genes,
    score_name="lyve1_core"
)
sc.pl.umap(adata, color = ['lyve1_core', 'anno_layer1'], size = 30, 
color_map = 'RdBu_r', save = 'umap_lyve1_core.png')

In [ ]:
sc.pl.dotplot(adata, var_names=['lyve1_core', 'LYVE1', 'MRC1', 'F13A1', 'FOLR2', 'SELENOP',], groupby="anno_layer1", standard_scale="var", 
cmap="RdBu_r", swap_axes=True,  dot_max= 0.5,  save = 'dotplot_lyve1_core.png')

In [ ]:
adata

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="anno_layer1", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=20, standard_scale='var', 
                                save = 'anno_layer1'
                               )

In [ ]:
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
     for group in groups for key in ['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']}
)
df.to_csv(f"{input_dir}/anno_layer1.csv", index=False)

In [ ]:
# Get top N marker genes per cluster from existing rank_genes_groups
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
n_top = 10

markers_dict = {group: list(result['names'][group][:n_top]) for group in groups}

# Same matrixplot styling as your functional plot
mp = sc.pl.matrixplot(
    adata,
    var_names=markers_dict,
    groupby="anno_layer1",
    standard_scale="var",
    cmap="RdBu_r",
    swap_axes=True,
    figsize=(7, 10),
    return_fig=True
)

mp.add_totals().style(edge_color="black")
mp.make_figure()

axes = mp.get_axes()
axes["mainplot_ax"].tick_params(axis="y", labelsize=4)
axes["mainplot_ax"].tick_params(axis="x", labelsize=8)

for ax in axes.values():
    for txt in ax.texts:
        txt.set_fontsize(4)
        txt.set_rotation(0)
        txt.set_ha("center")
        txt.set_va("center")

cax = axes["color_legend_ax"]
pos = cax.get_position()
cax.set_position([pos.x0 + 0.02, pos.y0, pos.width * 0.7, pos.height * 0.6])

plt.suptitle("Top 10 marker genes by cluster", fontsize=12, y=1)
plt.tight_layout()
plt.savefig("layer1_markers.png", bbox_inches="tight", dpi=600)
plt.show()

In [ ]:
markers_dict = {
    "LYVE1_signatures" : [
        'LYVE1', 'MRC1', 'F13A1', 'SELENOP', 'FOLR2', 'NRP1', 'CD68', 'C1QA'
    ],
    "oxphos_phosphorylation": [
        "COX4I1", "COX5B", "COX6B1", "COX7A2", "ATP5F1B",
        "ATP5F1C", "ATP5PO", "UQCRB", "NDUFA1", "SDHD"
    ],
    "antigen_presenting": [
        "CD74", "HLA-DRA", "HLA-DPA1", "HLA-DPB1", "HLA-DMA",
        "HLA-DMB", "HLA-DRB1", "FCER1G", "CTSS", "CD209"
    ],
    "actin_filament_organization": [
        "TMSB10", "TMSB4X", "AIF1", "TAGLN2", "PFN1",
        "CFL1", "ARPC1B", "ARPC3", "CDC42", "RAC1"
    ],
    "regulation_of_endocytosis": [
        "MERTK", "PICALM", "DOCK2", "RAB31", "AAK1",
        "SYK", "NUMB", "TLR2", "RIN3", "PLCG2"
    ],
    "clathrin_dependent_endocytosis": [
        "PICALM", "FCHO2", "ITSN1", "ITSN2", "AP2A2",
        "AAK1", "BMP2K", "FCHSD2", "SIGLEC1", "PIK3CB"
    ],
    "inflammatory_response": [
        "IL1B", "TNF", "CXCL8", "NFKBIA", "TNFAIP3",
        "NLRP3", "CCL2", "PTGS2", "TLR2", "CEBPB"
    ],
    "g2m_genes": [
        "MKI67", "TOP2A", "CENPF", "UBE2C", "BIRC5",
        "AURKB", "AURKA", "CDK1", "CDC20", "TPX2"
    ],
    "dna_replication": [
        "PCNA", "MCM2", "MCM5", "MCM6", "MCM7",
        "CDC6", "CDT1", "GINS1", "RRM2", "PCLAF"
    ],
    "detoxification":["HBA1", "HBA2", "HBB", "HBD", "HBG1", "HBG2",
    "HBM", "HBQ1", "HBZ"
        
    ],
    "regulation_of_apoptotic_signaling_pathway": [
        "LGALS3", "TREM2", "CTSS", "CTSH", "BAX",
        "IFI6", "TNFSF12", "FAIM", "HSPB1", "PARK7"
    ],
    "tumor_necrosis_factor_production": [
        "TREM2", "TYROBP", "GPNMB", "LPL", "CYBA",
        "LILRB4", "CD36", "SIRPA", "PTPN6", "LGALS9"
    ]
}

In [ ]:
mp = sc.pl.matrixplot(
    adata,
    var_names=markers_dict,
    groupby="anno_layer1",
    standard_scale="var",
    cmap="RdBu_r",
    swap_axes=True,
    figsize=(7, 8),
    return_fig=True
)

mp.add_totals().style(edge_color="black")
mp.make_figure()

axes = mp.get_axes()

axes["mainplot_ax"].tick_params(axis="y", labelsize=4)
axes["mainplot_ax"].tick_params(axis="x", labelsize=8)

for ax in axes.values():
    for txt in ax.texts:
        txt.set_fontsize(4)
        txt.set_rotation(0)
        txt.set_ha("center")
        txt.set_va("center")
cax = axes["color_legend_ax"]
pos = cax.get_position()
cax.set_position([pos.x0 + 0.02, pos.y0, pos.width * 0.7, pos.height * 0.6])

plt.suptitle("Marker genes by functional modules", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("layer1_function1.png", bbox_inches="tight", dpi = 600)
plt.show()

In [ ]:
sc.pl.umap(adata, color = ['NRP1', 'anno_layer1'], size = 30, color_map= 'RdBu_r', save = 'umap_anno_layer1.png', 
)

In [ ]:
import matplotlib.pyplot as plt
obs_df = adata.obs[['age', 'Disease']]
frequency_df = obs_df.groupby(['age', 'Disease']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across age / Disease")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
plt.savefig("dis_age.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
obs_df = adata.obs[['anatomical_site', 'anno_layer1']]
frequency_df = obs_df.groupby(['anatomical_site', 'anno_layer1']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across clusters / anatomical_site")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
plt.savefig("anno_layer1_anato.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
obs_df = adata.obs[['age', 'anno_layer1']]
frequency_df = obs_df.groupby(['age', 'anno_layer1']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across age / clusters")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
plt.savefig("freq_age.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Subset to adult samples only
adata_adult = adata[adata.obs['age'] == 'adult'].copy()

# Build percentage frequency table
obs_df = adata_adult.obs[['Disease', 'anno_layer1']]
frequency_df = obs_df.groupby(['Disease', 'anno_layer1']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

# Plot
fig, ax = plt.subplots(figsize=(10, 3))
percentage_df.plot(kind='barh', stacked=True, ax=ax)

ax.set_title("Cluster proportions across disease states (adults only)")
ax.set_xlabel("Percentage (%)")
ax.set_ylabel("Disease")
ax.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(False)
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig("freq_Disease_adult.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
obs_df = adata.obs[['anatomical_site', 'anno_layer1']]
frequency_df = obs_df.groupby(['anatomical_site', 'anno_layer1']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across age / clusters")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
plt.savefig("freq_anatomical_site.png", dpi=300)
plt.show()

In [ ]:
# pie charts for age. and also add n = .. for the clusters

In [ ]:
adata_healthy = adata[adata.obs["Disease"] == "healthy"].copy()

In [ ]:
adata_healthy

In [ ]:
adata_healthy.obs['Age_1'].unique()

In [ ]:
adata_pb = adata_healthy[adata_healthy.obs["Age_1"].isin(["adult", "feotus"])].copy()

In [ ]:
order = ["Mac0", "Mac1", "Mac2", "Mac3", "Mac4", "Mac5", "Mac6"]

adata_pb.obs["anno_layer1"] = pd.Categorical(
    adata_pb.obs["anno_layer1"],
    categories=order,
    ordered=True
)


import matplotlib.pyplot as plt
obs_df = adata_pb.obs[['Age_1', 'anno_layer1']]
frequency_df = obs_df.groupby(['Age_1', 'anno_layer1']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across age / clusters")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
plt.savefig("samplefreq.png", dpi=300)
plt.show()

In [ ]:
adata_pb = adata[adata.obs["age"] == "adult"].copy()

import matplotlib.pyplot as plt
obs_df = adata_pb.obs[['Disease', 'anno_layer1']]
frequency_df = obs_df.groupby(['Disease', 'anno_layer1']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(10, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters across Disease / clusters\n in adults")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
plt.savefig("adult_freq.png", dpi=300)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# =========================
# 1. Subset adult / foetus from healthy
# =========================
adata_pb = adata_healthy[adata_healthy.obs["age"].isin(["adult", "feotus"])].copy()

celltypes_use = ["heart_sc_Mac0", "heart_sc_Mac1", "heart_sc_Mac3", "heart_sc_Mac4"]

obs = adata_pb.obs[["sample", "age", "anno_layer1"]].copy()
sample_meta = obs[["sample", "age"]].drop_duplicates()

# =========================
# 2. Per-sample totals
# =========================
total_df = (
    obs.groupby(["sample", "age"])
    .size()
    .reset_index(name="total_cells")
)

# =========================
# 3. Per-celltype counts per sample
# =========================
count_df = (
    obs[obs["anno_layer1"].isin(celltypes_use)]
    .groupby(["sample", "age", "anno_layer1"])
    .size()
    .reset_index(name="n_cells")
)

# =========================
# 4. Fill missing sample-celltype combos with 0
# =========================
all_combos = (
    sample_meta.assign(key=1)
    .merge(pd.DataFrame({"anno_layer1": celltypes_use, "key": 1}), on="key")
    .drop(columns="key")
)

prop_df = (
    all_combos
    .merge(count_df, on=["sample", "age", "anno_layer1"], how="left")
    .merge(total_df, on=["sample", "age"], how="left")
)

prop_df["n_cells"] = prop_df["n_cells"].fillna(0)
prop_df["proportion"] = prop_df["n_cells"] / prop_df["total_cells"]

# Short label for plot
prop_df["celltype_short"] = prop_df["anno_layer1"].str.replace("heart_sc_", "")

# =========================
# 5. Sanity check
# =========================
print(prop_df.head())
print(prop_df.groupby(["age", "anno_layer1"])["proportion"].mean().unstack())

# =========================
# 6. Stats per celltype
# =========================
stats = []
for ct in celltypes_use:
    sub = prop_df[prop_df["anno_layer1"] == ct]
    g_adult = sub.loc[sub["age"] == "adult", "proportion"]
    g_foetus = sub.loc[sub["age"] == "feotus", "proportion"]

    if len(g_adult) < 3 or len(g_foetus) < 3:
        pval = np.nan
    else:
        _, pval = mannwhitneyu(g_adult, g_foetus, alternative="two-sided")

    stats.append({
        "celltype": ct,
        "adult_mean": g_adult.mean(),
        "foetus_mean": g_foetus.mean(),
        "pval": pval
    })

stats_df = pd.DataFrame(stats)
mask = stats_df["pval"].notna()
stats_df.loc[mask, "padj"] = multipletests(stats_df.loc[mask, "pval"], method="fdr_bh")[1]
stats_df["higher_in"] = stats_df.apply(
    lambda x: "adult" if x["adult_mean"] > x["foetus_mean"] else "foetus",
    axis=1
)
print(stats_df)

# =========================
# 7. Plot
# =========================
celltypes_short = [c.replace("heart_sc_", "") for c in celltypes_use]

fig, ax = plt.subplots(figsize=(7, 4))

sns.boxplot(
    data=prop_df, x="celltype_short", y="proportion", hue="age",
    order=celltypes_short, showfliers=False, ax=ax
)
sns.stripplot(
    data=prop_df, x="celltype_short", y="proportion", hue="age",
    order=celltypes_short, dodge=True, color="black", size=4,
    alpha=0.7, legend=False, ax=ax
)

ax.legend(title="age", bbox_to_anchor=(1.02, 1), loc="upper left")

# Annotate FDR
y_max = prop_df.groupby('celltype_short')['proportion'].quantile(0.95).max()
for i, ct in enumerate(celltypes_use):
    padj = stats_df.loc[stats_df["celltype"] == ct, "padj"].iloc[0]
    if pd.notna(padj):
        ax.text(i, y_max * 1.05, f"FDR={padj:.2e}", ha="center", fontsize=9)

ax.set_ylim(0, y_max * 1.20)
ax.set_xlabel("")
ax.set_ylabel("Proportion")
ax.set_title("Per-sample cell-type proportions (adult vs foetus, healthy donors)")

plt.tight_layout()
plt.savefig('stats.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import scanpy as sc

hb_hmox_panel = {
    "hemoglobin": ["HBB", "HBA1", "HBA2", "HBG1", "HBG2"],
    "heme_handling": ["HMOX1", "BLVRA", "CD163", "SPIC", "FTH1", "FTL"]
}

markers_flat = sum(hb_hmox_panel.values(), [])

# UMAP
sc.pl.umap(adata, color=markers_flat, ncols=4, cmap="RdBu_r", size=30,
           save="_hb_hmox_features.png")

# Dotplot
sc.pl.dotplot(adata, var_names=hb_hmox_panel, groupby="anno_layer1",
              standard_scale="var", cmap="RdBu_r", figsize=(8, 4),
              save="_hb_hmox_dotplot.png")

In [ ]:
sc.pl.dotplot(adata, var_names=hb_hmox_panel, groupby="anno_layer1",
              standard_scale="var", cmap="RdBu_r", figsize=(4, 4), swap_axes = True,
              save="_hb_hmox_dotplot.png")

In [ ]:
bulk50 = [
    "CCL3.AS1", "CCL18", "LYVE1", "F13A1", "CCL13", "LINC02688", "P2RY12",
    "SPATA8", "ZSWIM4", "MCF2L2P1", "ITGA9", "CCL23", "RAMP1", "P2RY14",
    "CCL26", "MIR6503", "GLDN", "CLEC4GP1", "SAA1", "TMIGD3", "IGHE",
    "HRH1", "PDK4", "CCDC89", "SOCS1", "MIR5581", "RBP4", "ZNF436",
    "ADAMTS2", "DAAM2", "GGT5", "IL17RB", "MED28P3", "ANGPTL4", "FKBP5",
    "MS4A6E", "MRC1", "VSIG4", "VCAM1", "C2", "EHD3", "C1QB", "MAOA",
    "ARC", "MANSC1", "CD200R1", "SESN1.1", "TFPI", "ENHO", "SLCO4A1"
]

sc.tl.score_genes(adata, gene_list=bulk50, score_name="bulk50_score")

In [ ]:
mac4_top50 = [
    "SNX29", "PDGFC", "IQGAP2", "JUNB", "DPYD", "LINC00278", "HRH1",
    "KLF4", "FOSB", "AL162414.1", "PMP22", "FHIT", "ELMO1", "F13A1",
    "FKBP5", "DUSP1", "MS4A4E", "ZFP36", "C1QA", "IER2", "JUN", "MYO5A",
    "TCF12", "C1QB", "MTSS1", "ZBTB20", "TBC1D5", "EPS15", "FTX",
    "ARHGAP15", "MED13L", "PLXDC2", "CELF2", "HDAC9", "AC007952.4",
    "STARD13", "LRMDA", "EXOC4", "TRPS1", "AP2A2", "DDX5", "SGMS1",
    "WDFY2", "FOXP1", "HS3ST2", "FOXN3", "PPM1L", "CERS6", "ARID1B",
    "SLC9A9"
]

mac4_top50 = [g for g in mac4_top50 if g in adata.var_names]
sc.tl.score_genes(adata, gene_list=mac4_top50, score_name="mac4_top50_score")

In [ ]:
sc.pl.umap(adata, color = ['HMOX1','anno_layer1'],  ncols= 2,
    size = 30, color_map = 'RdBu_r', #save = 'socres.png'
    )

In [ ]:
sc.pl.umap(adata, color = ['lyve1_core', 'bulk50_score', 'mac4_top50_score', 'anno_layer1'],  ncols= 2,
    size = 30, color_map = 'RdBu_r', save = 'socres.png')

In [ ]:
sc.pl.matrixplot(adata, var_names=["NRP1",'lyve1_core',"bulk50_score", "mac4_top50_score"],
                groupby = 'anno_layer1', cmap = 'RdBu_r', standard_scale = 'var',
                save = 'nrp1.png')

In [ ]:
hmox_mac_panel = {
    # Heme catabolism (core enzymes)
    "heme_catabolism": [
        "HMOX1",      # heme oxygenase 1 — rate-limiting, generates biliverdin+Fe+CO
        "HMOX2",      # constitutive isoform
        "BLVRA",      # biliverdin reductase A — biliverdin → bilirubin
        "BLVRB",      # biliverdin reductase B
    ],
    # Iron handling (post-heme breakdown)
    "iron_handling": [
        "FTH1",       # ferritin heavy chain — iron storage
        "FTL",        # ferritin light chain
        "SLC40A1",    # ferroportin — iron exporter
        "SLC11A1",    # NRAMP1 — endosomal iron transporter
        "HAMP",       # hepcidin
        "TFRC",       # transferrin receptor
    ],
    # Heme uptake (scavenger receptors)
    "heme_uptake": [
        "CD163",      # haptoglobin-hemoglobin scavenger receptor — KEY marker
        "CD163L1",    # paralogue
        "STAB1",      # stabilin-1 — scavenger
        "STAB2",      # stabilin-2
        "LRP1",       # alpha-2-macroglobulin / hemopexin receptor
    ],
    # Master TFs of heme-handling macrophage identity
    "TFs": [
        "SPIC",       # Spi-C — required for red pulp mac identity (Kohyama 2009)
        "BACH1",      # NRF2 antagonist, represses HMOX1
        "NFE2L2",     # NRF2 — induces HMOX1 + antioxidant response
        "KLF1",       # erythroid TF, already high in Mac4 per your data
        "MAFB",       # macrophage TF, partners with NRF2
    ],
    # Oxidative stress response (downstream of HMOX1 activity)
    "oxidative_stress": [
        "NQO1",       # NAD(P)H quinone dehydrogenase
        "GCLC",       # glutamate-cysteine ligase catalytic
        "GCLM",       # glutamate-cysteine ligase modifier
        "TXNRD1",     # thioredoxin reductase 1
        "SLC7A11",    # cystine/glutamate antiporter
        "PRDX1",      # peroxiredoxin 1
    ],
    # Erythrophagocytosis markers (red pulp macs)
    "erythrophagocytosis": [
        "VCAM1",      # adhesion, present on iron-recycling macs
        "MARCO",      # macrophage receptor with collagenous structure
        "TIMD4",      # T-cell Ig and mucin domain 4 — apoptotic cell recognition
        "MERTK",      # apoptotic cell receptor
        "GAS6",       # MERTK ligand
    ],
}

In [ ]:
sc.pl.dotplot(adata, var_names=hmox_mac_panel, groupby="anno_layer1", standard_scale="var", 
cmap="RdBu_r", swap_axes=True,  save = 'hmox')